# AirShift — Model Development & Comparison

## 1. Load the Labeled Dataset

The final labeled dataset created in the previous stage is loaded as the starting point for model development.

The dataset contains the engineered air quality and meteorological features together with the binary `Deterioration` target.

The labeled dataset is used without modifying the original saved file. Subsequent preprocessing and model development steps are performed on working copies of the data.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
# Define the data directories.
DATA_DIR = Path("../data/processed")
OUTPUT_DIR = Path("../data/processed")

# Load the final labeled dataset created in the labeling stage.
labeled_file = DATA_DIR / "airshift_labeled.csv"

df = pd.read_csv(
    labeled_file,
    parse_dates=["datetime"]
)

print("Dataset shape:", df.shape)
print("Number of stations:", df["station"].nunique())
print(
    "Date range:",
    df["datetime"].min(),
    "to",
    df["datetime"].max()
)

Dataset shape: (418381, 100)
Number of stations: 12
Date range: 2013-03-01 00:00:00 to 2017-02-28 17:00:00


In [3]:
# Check the final target distribution before model development.

print("Target distribution:")
print(df["Deterioration"].value_counts())

print("\nTarget distribution (%):")
print(
    df["Deterioration"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Target distribution:
Deterioration
0.0    219985
1.0    198396
Name: count, dtype: int64

Target distribution (%):
Deterioration
0.0    52.58
1.0    47.42
Name: proportion, dtype: float64


## 2. Define Features and Target

The dataset is separated into input features and the target variable required for supervised machine learning.

The `Deterioration` column is used as the binary target, while the remaining relevant variables are considered candidate input features.

Identifiers and columns that do not provide predictive information are excluded from the feature set.

The feature set will be further examined in the following leakage-audit stage before training the models.


In [4]:
# Define the target variable for binary classification.
TARGET = "Deterioration"

# Columns that should not be used as model features.
# "No" is an observation identifier, while "datetime" is handled
# separately for temporal splitting and is not directly used as a feature.
EXCLUDED_COLUMNS = [
    TARGET,
    "No",
    "datetime"
]

# Create the feature matrix and target vector.
X = df.drop(
    columns=EXCLUDED_COLUMNS
)

y = df[TARGET].astype(int)

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget values:")
print(sorted(y.unique()))

print("\nNumber of features:", X.shape[1])

Feature matrix shape: (418381, 97)
Target shape: (418381,)

Target values:
[np.int64(0), np.int64(1)]

Number of features: 97


## 3. Leakage Audit

Before training the machine learning models, the candidate features are examined for potential data leakage.

Data leakage occurs when a model receives information that would not have been available at the prediction time.

Because AirShift is an early-warning system, all model features must represent information available at or before the prediction time. Future-derived variables used during target construction must therefore be excluded from the feature set.

This audit checks the feature names for future-related information and confirms that the target variable is not included among the input features.


In [5]:
# Identify candidate features that may contain future-derived information.
# These features must not be used as model inputs.

future_keywords = [
    "future",
    "Deterioration"
]

potential_leakage_features = [
    column
    for column in X.columns
    if any(keyword.lower() in column.lower() for keyword in future_keywords)
]

print("Potential leakage features:")
print(potential_leakage_features)

Potential leakage features:
[]


In [6]:
# Confirm that the target variable is not present in the feature matrix.

print("Target included in X:", TARGET in X.columns)

Target included in X: False


In [7]:
# Check the dataset for any future-derived columns that should not
# be available as model features.

future_columns_in_dataset = [
    column
    for column in df.columns
    if "future" in column.lower()
]

print("Future-derived columns in dataset:")
print(future_columns_in_dataset)

Future-derived columns in dataset:
[]
